## Step 4 - CLIP image features

Encodes every keyframe with CLIP ViT-L/14-336 into an L2-normalised **768-d**
vector. Uses `get_image_features()` (the projected embedding that lives in the
same space as the text encoder) - NOT `vision_model.pooler_output`, which is the
un-projected 1024-d hidden state and is not comparable to text queries.

```
clip-features/<video_name>/<idx:06d>.npy   # shape (768,), float32, ||v|| = 1
```

In [ ]:
!pip -q install -U transformers tqdm

In [ ]:
import os
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, CLIPModel

In [ ]:
KEYFRAMES_DIR = '/kaggle/input/aic-keyframes-webp/Keyframes'   # output of Step 3
OUTPUT_DIR    = './clip-features'
MODEL_NAME    = 'openai/clip-vit-large-patch14-336'           # 768-d image features
BATCH_SIZE    = 64
IMAGE_EXTS    = ('.jpg', '.jpeg', '.png', '.webp')
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('device:', DEVICE)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print('loaded', MODEL_NAME)

In [ ]:
# Build the job list (flat layout, skip already-extracted .npy)
root, out_root = Path(KEYFRAMES_DIR), Path(OUTPUT_DIR)
jobs = []
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        out_path = out_root / img.relative_to(root).with_suffix('.npy')
        if not out_path.exists():
            out_path.parent.mkdir(parents=True, exist_ok=True)
            jobs.append((img, out_path))
print('to process:', len(jobs))

In [ ]:
@torch.no_grad()
def extract_batch(image_paths):
    imgs = [Image.open(p).convert('RGB') for p in image_paths]
    inputs = processor(images=imgs, return_tensors='pt').to(DEVICE)
    emb = model.get_image_features(**inputs)                  # (B, 768) projected
    emb = torch.nn.functional.normalize(emb, p=2, dim=-1)     # cosine-ready
    return emb.cpu().numpy().astype(np.float32)

In [ ]:
errors = []
for i in tqdm(range(0, len(jobs), BATCH_SIZE)):
    batch = jobs[i:i + BATCH_SIZE]
    try:
        vecs = extract_batch([p for p, _ in batch])
        for (_, out_path), v in zip(batch, vecs):
            np.save(out_path, v)                              # shape (768,), float32
    except Exception:
        for img_path, out_path in batch:                      # retry one by one
            try:
                np.save(out_path, extract_batch([img_path])[0])
            except Exception as e:
                errors.append((str(img_path), str(e)))
print('done. errors:', len(errors))
for p, e in errors[:10]:
    print(' ', p, '->', e)